In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score
import ot  # Python Optimal Transport library for Gromov-Wasserstein
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set(style="whitegrid")
%matplotlib inline

# Clustering Analysis: Feature-Based and Gromov-Wasserstein Distance

This notebook demonstrates two clustering approaches used in our analysis:

## 1. Feature-Based Clustering
- Uses extracted morphological features (19 features per sample)
- Applies t-SNE for dimensionality reduction and visualization
- Performs clustering on feature vectors

## 2. Gromov-Wasserstein Distance (GWD) Based Clustering
- Computes pairwise Gromov-Wasserstein distances between graph structures
- Uses distance matrix for hierarchical clustering
- Visualizes results with t-SNE on distance matrix

**Note**: For demonstration, we will use feature data. For full GWD analysis, you would need the original graph structures (.pkl files).


In [ ]:
# Load the feature dataset
# Replace with your actual feature file path
# Options: 'main_neuron_features.csv', 'main_astro_features.csv', 'main_road_network_features.csv'

try:
    df = pd.read_csv('../demo/main_neuron_features.csv')
    print(f"Loaded dataset successfully: {df.shape[0]} samples, {df.shape[1]} columns")
except FileNotFoundError:
    print("Dataset not found. Please run the feature extraction notebooks in the 'demo' folder first.")
    # For demo purposes, create synthetic data
    from sklearn.datasets import make_blobs
    X_dummy, y_dummy = make_blobs(n_samples=300, centers=5, n_features=19, random_state=42)
    feature_names = ['N_edg', 'Betw', 'Spec_ent', 'Alg_Conn', 'Assort', 'G_diam', 
                     'Br_dens', 'Br_l', 'Energy', 'Apl', 'Circ', 'FD', 'Dir_std',
                     'q1', 'q2', 'q3', 'q4', 'Dir_ent', 'Order']
    df = pd.DataFrame(X_dummy, columns=feature_names)
    df['label'] = y_dummy
    label_map = {0: 'Basal_ganglia', 1: 'Cerebellum', 2: 'Hippocampus', 
                 3: 'Olfactory_bulb', 4: 'Retina'}
    df['label'] = df['label'].map(label_map)
    df['id'] = [f'sample_{i}' for i in range(len(df))]

# Display basic information
print(f"\nLabel distribution:")
print(df['label'].value_counts())
df.head()

In [ ]:
# Separate features and labels
metadata_cols = ['label', 'id', 'region_name', 'type']
feature_cols = [c for c in df.columns if c not in metadata_cols]

X = df[feature_cols].values
y = df['label'].values if 'label' in df.columns else None
sample_ids = df['id'].values if 'id' in df.columns else np.arange(len(X))

# Handle missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Normalize features (z-score normalization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Number of classes: {len(np.unique(y)) if y is not None else 'Unknown'}")

---
## Part 1: Feature-Based Clustering

We cluster samples based on their extracted morphological features.


In [ ]:
# Apply t-SNE for visualization
print("Computing t-SNE (this may take a minute)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_scaled)-1), 
            n_iter=1000, learning_rate=200)
X_tsne = tsne.fit_transform(X_scaled)

# Create DataFrame for plotting
tsne_df = pd.DataFrame({
    'Dim 1': X_tsne[:, 0],
    'Dim 2': X_tsne[:, 1],
    'Label': y if y is not None else 'Unknown'
})

# Plot t-SNE with true labels
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
for label in np.unique(y):
    mask = y == label
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=label, s=100, alpha=0.7)
ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('t-SNE Visualization: Feature-Based', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# K-Means clustering on features
n_clusters = len(np.unique(y)) if y is not None else 5
print(f"Performing K-Means clustering with k={n_clusters}...")

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels_kmeans = kmeans.fit_predict(X_scaled)

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels_kmeans, 
                     cmap='tab10', s=100, alpha=0.7)
ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('K-Means Clustering Results (Feature-Based)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.show()

# Evaluation
sil_score = silhouette_score(X_scaled, cluster_labels_kmeans)
print(f"\nSilhouette Score: {sil_score:.3f}")

if y is not None:
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    ari = adjusted_rand_score(y_encoded, cluster_labels_kmeans)
    print(f"Adjusted Rand Index (ARI): {ari:.3f}")

---
## Part 2: Gromov-Wasserstein Distance (GWD) Based Clustering

In the full analysis, GWD measures structural similarity between graphs without requiring node correspondence.
Here we demonstrate the workflow using feature-based distance as a proxy.

**Note**: Computing true GWD between all pairs of large graphs is computationally expensive. 
In practice, you would load pre-computed distance matrices or compute them from graph structures.


In [ ]:
# For demonstration, we compute a distance matrix
# In real GWD analysis, this would be computed from graph structures using ot.gromov_wasserstein2()

# Option 1: Use Euclidean distance as proxy (for demo)
from scipy.spatial.distance import pdist, squareform
dist_matrix = squareform(pdist(X_scaled, metric='euclidean'))

print(f"Distance matrix shape: {dist_matrix.shape}")
print(f"Distance range: [{dist_matrix.min():.3f}, {dist_matrix.max():.3f}]")

# Visualize distance matrix
plt.figure(figsize=(8, 6))
sns.heatmap(dist_matrix[:50, :50], cmap='viridis', square=True, cbar_kws={'label': 'Distance'})
plt.title('Pairwise Distance Matrix (first 50 samples)', fontsize=14, fontweight='bold')
plt.xlabel('Sample Index')
plt.ylabel('Sample Index')
plt.tight_layout()
plt.show()

In [ ]:
# Hierarchical clustering using the distance matrix
print(f"Performing Agglomerative Clustering with k={n_clusters}...")

agg_clustering = AgglomerativeClustering(n_clusters=n_clusters, metric='precomputed', 
                                          linkage='average')
cluster_labels_gwd = agg_clustering.fit_predict(dist_matrix)

# t-SNE on distance matrix for visualization
print("Computing t-SNE on distance matrix...")
tsne_gwd = TSNE(n_components=2, random_state=42, metric='precomputed', 
                perplexity=min(30, len(dist_matrix)-1), n_iter=1000)
X_tsne_gwd = tsne_gwd.fit_transform(dist_matrix)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: True labels
for label in np.unique(y):
    mask = y == label
    axes[0].scatter(X_tsne_gwd[mask, 0], X_tsne_gwd[mask, 1], label=label, s=100, alpha=0.7)
axes[0].set_xlabel('t-SNE Dimension 1', fontsize=12)
axes[0].set_ylabel('t-SNE Dimension 2', fontsize=12)
axes[0].set_title('t-SNE on Distance Matrix: True Labels', fontsize=14, fontweight='bold')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 2: Predicted clusters
scatter = axes[1].scatter(X_tsne_gwd[:, 0], X_tsne_gwd[:, 1], c=cluster_labels_gwd, 
                          cmap='tab10', s=100, alpha=0.7)
axes[1].set_xlabel('t-SNE Dimension 1', fontsize=12)
axes[1].set_ylabel('t-SNE Dimension 2', fontsize=12)
axes[1].set_title('Hierarchical Clustering (GWD-Based)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluation of GWD-based clustering
# So we use ARI to compare with ground truth

if y is not None:
    ari_gwd = adjusted_rand_score(y_encoded, cluster_labels_gwd)
    print(f"Adjusted Rand Index (GWD-Based): {ari_gwd:.3f}")
    print(f"\nComparison:")
    print(f"  Feature-Based Clustering ARI: {ari:.3f}")
    print(f"  GWD-Based Clustering ARI:     {ari_gwd:.3f}")

---
## Summary

We demonstrated two clustering approaches:

1. **Feature-Based Clustering**: 
   - Direct clustering on extracted morphological features
   - Works well when features capture structural differences

2. **Gromov-Wasserstein Distance (GWD) Based Clustering**:
   - Measures structural similarity between graphs
   - More computationally expensive but captures topological relationships
   - Particularly useful when graph structure matters more than individual features

Both approaches can complement each other in analyzing complex network data.
